In [18]:
import json
import os
from transformers import AutoTokenizer

/Users/user/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Library/Developer/CommandLineTools/Library/Framew

In [19]:
INPUT_PATH = "data/literature/SNCA_A53T_corpus.json"
OUTPUT_PATH = "data/literature/chunked_corpus.json"

MODEL_NAME = "pritamdeka/S-PubMedBert-MS-MARCO"

print(f"⚙️ Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

⚙️ Loading tokenizer: pritamdeka/S-PubMedBert-MS-MARCO


tokenizer_config.json:   0%|          | 0.00/388 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [21]:
with open(INPUT_PATH, "r") as f:
    papers = json.load(f)

print(f"📚 Loaded papers: {len(papers)}")

📚 Loaded papers: 100


In [22]:
def chunk_by_tokens(text, tokenizer, chunk_size=250, overlap=40):
    """
    Token-based chunking aligned with embedding model.
    Ensures chunks stay within transformer limits.
    """
    if not text or not text.strip():
        return []
    
    tokens = tokenizer.encode(text, add_special_tokens=False)
    
    chunks = []
    step = chunk_size - overlap
    
    for i in range(0, len(tokens), step):
        token_window = tokens[i : i + chunk_size]
        
        # Skip very small chunks (low quality)
        if len(token_window) < 50:
            continue
        
        chunk_text = tokenizer.decode(token_window, skip_special_tokens=True)
        chunks.append(chunk_text)
        
        if i + chunk_size >= len(tokens):
            break
            
    return chunks

In [23]:
chunked_data = []

for paper in papers:
    pmid = paper.get("pmid")
    title = paper.get("title", "")
    abstract = paper.get("abstract", "")
    
    if not abstract:
        continue
    
    chunks = chunk_by_tokens(abstract, tokenizer)
    
    for chunk in chunks:
        chunked_data.append({
            "pmid": pmid,
            "title": title,
            "chunk": chunk
        })

print(f"🧩 Total chunks created: {len(chunked_data)}")

🧩 Total chunks created: 142


In [24]:
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

with open(OUTPUT_PATH, "w") as f:
    json.dump(chunked_data, f, indent=4, ensure_ascii=False)

print(f"✅ Saved chunked corpus to {OUTPUT_PATH}")

✅ Saved chunked corpus to data/literature/chunked_corpus.json


In [25]:
print("Total chunks:", len(chunked_data))

# Check chunk size
print("Sample chunk length:", len(chunked_data[0]["chunk"].split()))

# Ensure structure
print(chunked_data[0])

Total chunks: 142
Sample chunk length: 201
{'pmid': '41002401', 'title': 'Modeling Synucleinopathy Using hESC-Derived Cerebral Organoids.', 'chunk': 'animal and cellular models harboring snca gene mutations have been instrumental in synucleinopathy, but faithful human brain models remain limited. here, we report the development of a human cerebral organoid ( co ) model of synucleinopathy carrying the ala53thr mutation in snca ( snca a53t ). using a human embryonic stem cell ( hesc ) line overexpressing snca a53t ( a53t hesc line ), we generated cos ( a53t cos ) that recapitulate hallmark features of synucleinopathy. these a53t cos exhibited elevated α - synuclein ( α - syn ) expression, the increased phosphorylation of α - syn, and lewy body - like aggregations. notably, we also observed the increased expression of phosphorylated tau and neurofibrillary tangle - like silver deposits, although amyloid β expression and accumulation remained unchanged. to evaluate the utility of this mode

In [26]:
# Average token length
avg_len = sum(len(tokenizer.encode(c["chunk"])) for c in chunked_data) / len(chunked_data)
print(f"📏 Avg tokens per chunk: {avg_len:.2f}")

# Ensure no empty chunks
print("Empty chunks present:",
      any(len(c["chunk"].strip()) == 0 for c in chunked_data))

📏 Avg tokens per chunk: 197.43
Empty chunks present: False
